In [9]:
import boto3
import json
import os
import pandas as pd
from dotenv import load_dotenv

load_dotenv()

True

In [10]:
df = pd.read_csv('../Dataset/news_excerpts_parsed.csv')
df = df[df["text"].str.contains("terrorism", case=False)]
df

,link,text
38,https://www.channelnewsasia.com/singapore/mas-...,The Monetary Authority of Singapore (MAS) has ...
119,https://www.channelnewsasia.com/cna-insider/wh...,SINGAPORE: Southeast Asia need not worry about...
833,https://www.channelnewsasia.com/singapore/sing...,Singapore and the Philippines are looking to b...
936,https://www.channelnewsasia.com/singapore/man-...,"Kevin Francis Hawkins, 30, was suffering from ..."
1223,https://www.channelnewsasia.com/asia/unusual-m...,"""China offered to support long-time strategic ..."
1367,https://edition.cnn.com/2011/09/30/politics/ta...,The U.S. drone killing of American-born and -r...
1492,https://www.straitstimes.com/business/mas-impo...,The Monetary Authority of Singapore (MAS) has ...


In [14]:
# Define the template for Bedrock request
def generate_bedrock_request(query: str) -> str:
    return (json.dumps
        ({
        "anthropic_version": "bedrock-2023-05-31",
        "max_tokens": 1000,
        "system": "You are an assistant helping Singapore's Internal Security Department (ISD) in addressing four critical areas of concern: Espionage, Terrorism and Violent Extremism, Cybersecurity, and Communalism. Using datasets from WikiLeaks articles and news excerpts, the goal is to extract structured data to identify key entities, relationships, and incidents relevant to Singapore’s internal security. \
The data may include unrelated global events (e.g., the Taliban takeover in Afghanistan), but these are still valuable for identifying patterns, trends, and lessons that could impact Singapore indirectly. By scoring relevance and extracting actionable insights, ISD can prioritize risks, allocate resources effectively, and maintain a proactive stance against both domestic and international security threats.",
        "messages": [
            {
                "role": "user",
                "content":
                    f"Provide the output of the text in the format: "
                    f"- Entity "
                    f"- Type (Person, Organization, Location, Incident, Keyword) "
                    f"- Date (if mentioned) "
                    f"- Summary (brief overview of relevance to ISD's concerns)"
                    f"- Area of Concern (Espionage, Terrorism, Cybersecurity, Communalism)"
                    f"- Relevance Score (High, Medium, Low)"
                    f"- Relationships (if applicable, e.g., Person A linked to Organization B)."
                    f"Here is the text:"
                    f"{query}"
            }
        ]
    }
    ))

In [15]:
json.loads(generate_bedrock_request(df.iloc[0]['text']))

{'anthropic_version': 'bedrock-2023-05-31',
 'max_tokens': 1000,
 'messages': [{'role': 'user',
   'content': 'Provide the output of the text in the format: - Entity - Type (Person, Organization, Location, Incident, Keyword) - Date (if mentioned) - Summary (brief overview of relevance to ISD\'s concerns)- Area of Concern (Espionage, Terrorism, Cybersecurity, Communalism)- Relevance Score (High, Medium, Low)- Relationships (if applicable, e.g., Person A linked to Organization B).Here is the text:The Monetary Authority of Singapore (MAS) has fined banks DBS, OCBC, Citibank and insurer Swiss Life a total of S$3.8 million (US$2.83 million) for breaching its requirements on anti-money laundering and countering terrorism financing.\n\nThe breaches were identified when MAS examined the financial institutions following news of irregularities relating to Wirecard AG’s financial statements and the alleged involvement of Singapore-based individuals and entities.\n\nThe financial institutions were

In [16]:
client = boto3.client(
    'bedrock-runtime',
    aws_access_key_id=os.environ["AWS_ACCESS_KEY_ID"],
    aws_secret_access_key=os.environ["AWS_SECRET_ACCESS_KEY"],
    region_name='ap-southeast-1'
)
response = client.invoke_model(
    body=generate_bedrock_request(df.iloc[0]['text']),
    modelId='anthropic.claude-3-haiku-20240307-v1:0',
    contentType='application/json',
    accept='application/json'
)

ClientError: An error occurred (InvalidSignatureException) when calling the InvokeModel operation: The request signature we calculated does not match the signature you provided. Check your AWS Secret Access Key and signing method. Consult the service documentation for details.